# Susceptibilidad de incendios — entrenamiento en Colab
Ejecuta en orden. Compara RF y MLP con atención usando un CSV de covariables ya extraídas. Evaluación aleatoria, no espacial. CPU es suficiente; no necesita montar todo tu Drive.


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile, shutil
from google.colab import files

# Después de publicar, pega la URL HTTPS de tu repositorio. Vacío: cargar ZIP.
REPO_URL = "https://github.com/javier150381/susceptibilidad-incendios-ecuador.git"
ROOT = Path('/content/incendios-colab')
if not ROOT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    else:
        print('Carga incendios-colab.zip')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Carga exactamente un ZIP.')
        name = next(iter(uploaded))
        with zipfile.ZipFile(name) as archive:
            base = Path('/content').resolve()
            for member in archive.infolist():
                if not (base / member.filename).resolve().is_relative_to(base):
                    raise ValueError('Ruta inválida en ZIP')
            archive.extractall(base)
        del uploaded
if not (ROOT / 'src/train.py').is_file():
    raise FileNotFoundError('No se encontró src/train.py en el paquete/repositorio.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)


## Cargar datos
Selecciona el CSV `dataset_entrenamiento_sierra_realista.csv` de la carpeta `resultados` original. Esta carga no publica tus datos en GitHub.


In [ ]:
print('Carga el CSV de entrenamiento')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Carga exactamente un CSV.')
name = next(iter(uploaded))
DATA = ROOT / 'data/dataset_entrenamiento_sierra_realista.csv'
DATA.parent.mkdir(parents=True, exist_ok=True)
DATA.write_bytes(uploaded[name])
del uploaded
import pandas as pd
df = pd.read_csv(DATA)
display(df.head())
print('Filas:', len(df))
print(df['clase'].value_counts())


In [ ]:
import os
env = os.environ.copy()
env['FIRE_DATA'] = str(DATA)
env['FIRE_OUTPUT'] = str(ROOT / 'results')
env['FIRE_EPOCHS'] = '50'  # Usa 1 solo para una prueba de ejecución.
env['MPLBACKEND'] = 'Agg'
subprocess.run([sys.executable, str(ROOT / 'src/train.py')], env=env, check=True)
display(pd.read_csv(ROOT / 'results/metricas.csv'))
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'results/comparativa_modelos_factores_realista.png')))


## Guardar resultados
Descarga el ZIP: el almacenamiento de la sesión de Colab es temporal. Consulta el estado metodológico antes de interpretar métricas o factores.


In [ ]:
bundle = shutil.make_archive('/content/resultados-incendios', 'zip', ROOT / 'results')
files.download(bundle)
